# 01 · Configuration, population and outcome definition — Fetal death, MIMIC-IV (M3)

| Field | Content |
|---|---|
| Framework sequence | **01: S1 · S2 · S3 · G1** → 02: S4 → 03: S5 · S6 → R3 → 04: R3 · S2 · S3 · G1 · S4 → 05: S5 · S6 · S7 · G2 → 06: G3 · class · S11 |
| What it does | Shows the configuration (S1), counts the obstetric universe and the population (S2), verifies the catalogue of fetal-death codes and the label table by admission (S3), and checks that the reference counts of the count audit are reproduced (G1) |
| Stages and gates | S1 · S2 · S3 · G1 |
| Decisions | D1 (database), D3 (population), D5 (outcome definition); see `docs/outcome_card_fetal_death.md` |
| Inputs | MIMIC-IV v3.1 in PostgreSQL and the revised obstetric cohort (`cohort_obstetric_v2` and companion tables) |
| Tables of the mimiciv schema | `fetal_death_code_patterns_v2` (`sql/01_labels/01_fetal_death_codes_official.sql`) and `fetal_death_label_v2` (`sql/01_labels/02_fetal_death_label_by_admission.sql`) |
| Outputs | `data/outputs_01_outcome_definition.txt` (aggregate counts only; 1 to 4 shown as "<5") |
| Time | About a minute when the tables exist |
| Previous / next | — / `02_count_audit` |
| Spanish version | `01_outcome_definition_es.ipynb` (same code) |

> **Rebuild the tables.** The tables of the `mimiciv` schema that this notebook needs are read as they are, and only their checks run. If a table does not exist, it is built from its SQL script. To rebuild it (for example after a change in a script of `sql/`), set **`REBUILD = True`** in the first code cell and run the notebook again.

In [ ]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_01_outcome_definition.txt"
engine = get_connect_database()

## S1 · Governance and configuration — source, data use agreement, seed, software and publication rule

Fixed before running any analysis: the data source and its version, the data use agreement, the random seed, the software versions, the reference counts to be reproduced and the publication rule. The cell shows that configuration and checks that the database answers, that the tables of the revised obstetric cohort exist and that the packages are installed.

| Result | What is done | In this model |
|---|---|---|
| The database answers, the cohort tables exist and the packages are installed (ok) | On to S2 | PostgreSQL copy of MIMIC-IV v3.1 |
| No connection to the database | Check `.env` and that PostgreSQL is running (README) | — |
| A cohort table is missing | Build the revised obstetric cohort first (README, prerequisites) | — |

In [ ]:
# Stage S1: the configuration every notebook of this repository uses, shown so it can be audited.
import platform
def version(module):
    try:
        return importlib.import_module(module).__version__
    except Exception:
        return "NOT INSTALLED"
db_error = ""
try:
    with engine.connect() as _con:
        _con.exec_driver_sql("SELECT 1")
    db_status = "ok"
except Exception as exc:                      # keep the cause, so the message says what failed
    db_status = "NO CONNECTION: check .env and that PostgreSQL is running"
    db_error = f"{type(exc).__name__}: {(str(exc).strip().splitlines() or [''])[0][:200]}"
UPSTREAM = ["cohort_obstetric_v2", "hadm_delivery_v2", "obstetric_dx_v2", "predictors_24h_v2"]
up_missing = [t for t in UPSTREAM if db_status != "ok" or not table_exists(engine, t)]
rows = [("data source", "MIMIC-IV v3.1 (PhysioNet), Beth Israel Deaconess Medical Center, 2008-2022; dates shifted per patient", "fixed"),
        ("database", "local PostgreSQL copy of MIMIC-IV (schema mimiciv); credentials in .env (not versioned)", db_status),
        ("revised obstetric cohort", ", ".join(UPSTREAM), "ok" if not up_missing else "MISSING: " + ", ".join(up_missing)),
        ("data use", "PhysioNet Credentialed Health Data Use Agreement and CITI training: research only; no re-identification; "
                     "data not shared or redistributed", "rule"),
        ("publication rule", "only aggregate outputs (data/outputs_*.txt); counts from 1 to 4 shown as <5 (D13); "
                             "no subject_id or hadm_id published", "rule"),
        ("random seed", "20261119 (notebooks 05 and 06)", "fixed"),
        ("reference counts", "frozen at the count audit (count_audit.md) and reproduced by gate G1 in notebooks 01 and 04", "fixed"),
        ("python", platform.python_version(), "info")]
if db_error:
    rows.insert(2, ("database error", db_error, "check"))
rows += [(m, version(m), "ok" if version(m) != "NOT INSTALLED" else "INSTALL (requirements.txt)")
         for m in ["pandas", "numpy", "sklearn", "scipy", "sqlalchemy", "psycopg2"]]
show("S1 · Configuration of this repository", pd.DataFrame(rows, columns=["item", "value", "status"]))
assert db_status == "ok" and not up_missing, (tr("S1: fix the configuration before going on")
    + f" | {db_status}{' (' + db_error + ')' if db_error else ''} | " + ", ".join(up_missing))

## S2 · Universe and population — obstetric admissions and population of the model (decision D3)

The universe is the revised obstetric cohort: admissions of women aged 12 to 55 with a diagnosis of pregnancy, delivery or puerperium. The population of the model is the delivery hospitalizations plus the admissions with a stillbirth code that the delivery flag left out (decision D3).

| Result | What is done | In this model |
|---|---|---|
| The counts match the count audit | The population is fixed: on to S3 | 19,440 admissions of 13,735 patients |
| A count differs | Review the revised obstetric cohort before going on | — |

In [ ]:
# Stage S2: obstetric universe and population of the model (decision D3), on the revised obstetric cohort
pop = pd.read_sql("""
    SELECT COUNT(*)                                                               AS n_admissions,
           COUNT(DISTINCT subject_id)                                             AS n_patients,
           COUNT(*) FILTER (WHERE is_delivery_v2)                                 AS n_delivery_hospitalizations,
           COUNT(*) FILTER (WHERE has_stillbirth_code AND NOT is_delivery_v2)     AS n_stillbirth_code_without_delivery_flag,
           COUNT(*) FILTER (WHERE is_delivery_v2 OR has_stillbirth_code)          AS n_population,
           COUNT(DISTINCT subject_id) FILTER (WHERE is_delivery_v2 OR has_stillbirth_code) AS n_patients_population
    FROM   mimiciv.cohort_obstetric_v2""", engine)
show("S2 · Obstetric universe and population of the model", pop)
p0 = pop.iloc[0]
decision("S2 · Decision D3: population", [
    ("population of the model", f"{int(p0.n_population):,} admissions of {int(p0.n_patients_population):,} patients",
     "delivery hospitalizations (v2 flag) or a stillbirth code"),
    ("stillbirth codes without the delivery flag", f"{int(p0.n_stillbirth_code_without_delivery_flag)} admissions",
     "they stay in the population: the delivery flag leaves them out because they also carry an abortive code"),
    ("record and grouping", "one row per admission", "one record per delivery hospitalization, grouped by patient (S5, notebook 03)")])

## S3 · Outcome definition — verified catalogue of fetal-death codes (task T15)

The outcome is a proxy: a fetal-death diagnosis code in the mother's admission. MIMIC-IV has no mother-newborn link, so early neonatal deaths are not captured. The catalogue lists every code pattern with its official title and its tier: *strict* (fetal death of the whole pregnancy), *partial* (a multiple pregnancy in which some fetuses died) and *excluded* (early fetal death and codes of the newborn record, listed only for audit).

| Result | What is done | In this model |
|---|---|---|
| The catalogue has the 18 expected patterns and no strict code falls outside the universe | The catalogue is fixed: the label table is built with it | Strict 4 + 4, partial 3 + 2, excluded 2 + 3 (ICD-10 + ICD-9) |
| A pattern is missing or a code falls outside the universe | Correct the catalogue (`sql/01_labels/01_...`) and rebuild | — |

In [ ]:
# Stage S3: verified catalogue of fetal-death codes (task T15): mimiciv.fetal_death_code_patterns_v2
res15 = ensure_table(engine, "fetal_death_code_patterns_v2", SQL / "01_labels" / "01_fetal_death_codes_official.sql",
                     REBUILD, log=say)
show_results(res15, ["Q-T15a patterns by tier", "Q-T15b admissions per pattern", "Q-T15c full codes in the universe",
                     "Q-T15e excluded families in the universe"])

## S3 · Outcome definition — label table by admission (task T16) and decision D5

One row per admission of the universe. Besides the main label, it keeps every sensitivity definition as a column (partial codes, 22 and 28 weeks or more, without early-loss codes), the position of the code, the gestational weeks of the Z3A codes and the ICD era. The output ends with the table «Decision D5», which shows the result of the review and the path taken.

| Result | What is done | In this model |
|---|---|---|
| Gestational weeks only in ICD-10 and pre-viable losses inside the label | Main label: strict code at any gestational age; the rest as sensitivity | D5: 129 strict events; 142 with partial codes; 55 with 22 weeks or more |
| The label has no event time | Recorded as a known risk: the timing of the events is measured in S6 (notebook 03) | 65 of 129 events have the fetal-death code as principal diagnosis |

In [ ]:
# Stage S3: label table by admission (task T16): mimiciv.fetal_death_label_v2
res16 = ensure_table(engine, "fetal_death_label_v2", SQL / "01_labels" / "02_fetal_death_label_by_admission.sql",
                     REBUILD, log=say)
show_results(res16, ["Q-T16b strict events outside population D3", "Q-T16c strict events by code family",
                     "Q-T16d position of the strict code", "Q-T16e events with early-loss code", "Q-T16f label sizes",
                     "Q-T16g D8 admissions shorter than 6 h"])
sizes = res16["Q-T16f label sizes"].set_index("scope").iloc[0]
decision("S3 · Decision D5: outcome definition", [
    ("main label", f"{int(sizes.n_main_strict)} strict events", "strict fetal-death code at any gestational age"),
    ("strict plus partial codes", f"{int(sizes.n_strict_plus_partial)} events", "sensitivity analysis"),
    ("strict without an early-loss code", f"{int(sizes.n_strict_without_early_loss_code)} events", "sensitivity analysis"),
    ("22 weeks or more (ICD-10 only)", f"{int(sizes.n_ge22_weeks)} events", "sensitivity analysis"),
    ("28 weeks or more (ICD-10 only)", f"{int(sizes.n_ge28_weeks)} events", "sensitivity analysis"),
    ("strict events without gestational weeks", f"{int(sizes.n_strict_weeks_unknown)} events", "reported apart"),
    ("time of the event", "none: discharge codes without a date", "measured with proxies in S6 (notebook 03)")])

## G1 · Definition gate — are the reference counts reproduced?

Population, unit and outcome are fixed and the counts can be reproduced. The reference counts are those of the count audit (`count_audit.md`). If any differs, the notebook stops.

| Result | What is done | In this model |
|---|---|---|
| Every count is reproduced | G1 passes: the definition is fixed; on to the count audit (notebook 02) | Passes |
| A count differs | It stops (`assert`): go back to S2 or S3 until the definition is stable | — |

In [ ]:
# Gate G1: reference counts of the count audit (count_audit.md, sections 3.1, 3.6 and 3.7)
t15 = res15["Q-T15d events with the verified table"].iloc[0]
t16 = res16["Q-T16a integrity"].iloc[0]
g1_ok = check([
    ("universe: obstetric admissions", int(p0.n_admissions), 25736),
    ("universe: patients", int(p0.n_patients), 15084),
    ("delivery hospitalizations", int(p0.n_delivery_hospitalizations), 19425),
    ("label table: rows (one per admission)", int(t16.n_rows), 25736),
    ("label table: distinct admissions", int(t16.n_distinct_admissions), 25736),
    ("events, strict or partial (universe)", int(t16.n_events), 154),
    ("strict events (universe)", int(t16.n_strict), 136),
    ("partial-only events (universe)", int(t16.n_partial_only), 18),
    ("population D3", int(t16.n_population_d3), 19440),
    ("population D3 (S2) = population of the label table", int(p0.n_population), int(t16.n_population_d3)),
    ("events in population D3", int(t16.n_events_population_d3), 142),
    ("strict events in population D3 (main label)", int(t16.n_strict_population_d3), 129),
    ("strict events outside population D3", int(t16.n_strict_outside_population), 7),
    ("events with an early-loss code", int(t16.n_events_with_early_loss_code), 11),
    ("verified catalogue: strict events (universe)", int(t15.n_strict), 136),
    ("verified catalogue: strict events in population D3", int(t15.n_strict_population_d3), 129),
], "G1 · Reference counts of the count audit")
decision("G1 · Definition gate", [
    ("population, unit and outcome fixed; counts reproducible", "passes" if g1_ok else "DOES NOT PASS",
     "the definition is fixed: on to the count audit (notebook 02)" if g1_ok else "stop: go back to S2 or S3")])
assert g1_ok, "gate G1 does not pass: a reference count differs"

## Notebook summary

Main indicators of this notebook and the next step.

In [ ]:
# Summary of notebook 01
notebook_summary([
    ("population of the model (D3)", lambda: f"{int(p0.n_population):,} admissions of {int(p0.n_patients_population):,} patients"),
    ("main label (D5)", "strict fetal-death code at any gestational age"),
    ("strict events in the population", lambda: f"{int(t16.n_strict_population_d3)} ({100 * int(t16.n_strict_population_d3) / int(t16.n_population_d3):.2f}%)"),
    ("events with partial codes (sensitivity)", lambda: int(t16.n_events_population_d3)),
    ("event time in the data", "none: discharge codes without a date"),
    ("gate G1", lambda: "passes" if g1_ok else "DOES NOT PASS"),
    ("next step", "count audit (S4), notebook 02"),
])
write_outputs(OUT_FILE, "Notebook 01 (S1, S2, S3, G1), fetal death MIMIC-IV (M3); aggregate output only")